# m25 quenched sample — 5-arm RT campaign (almac11-style)

Staging companion of `powderday_flux_quenched_m25.ipynb`: extends the quenched-cis25
mock observations to the same arm structure as the ALMA-C11 campaign
(`analogues_specphot_almac11_rt.ipynb`) — dust on/off, AGN on/off, and the CLUMPY
torus viewed at three inclinations.

| arm | tree (`run_tag`) | master | AGN | sightlines | who | status |
|---|---|---|---|---|---|---|
| `dust_on` | `dusty_simdust` | `parameters_master.py` | off | 4 | all | **existing** |
| `dust_off` | `nodust_1e-12` | `parameters_master-nodust.py` | off | 4 | all | **existing** |
| `agn_hopkins` | `dusty_simdust_agn` | `parameters_master-agn.py` | Hopkins+2007 | 4 | BH hosts | **existing** |
| `nenkova_i90` | `nenkova_i90` | `parameters_master-nenkova-i90.py` | CLUMPY i=90 | 1 | BH hosts | **NEW** |
| `nenkova_i60` | `nenkova_i60` | `parameters_master-nenkova-i60.py` | CLUMPY i=60 | 1 | BH hosts | **NEW** |
| `nenkova_i30` | `nenkova_i30` | `parameters_master-nenkova-i30.py` | CLUMPY i=30 | 1 | BH hosts | **NEW** |

**Locked decisions**

- The three existing trees under `output/cis25/sed_quenched_regions/` are NEVER
  restaged here — Part 2 only takes a `sed_is_complete` census of them; incomplete
  SEDs mean their own `submit_all_snaps.sh` still needs (re)running.
- The masters are the SAME files the almac11 campaign uses (`simbanator/sed/`),
  so arm homogeneity is machine-checked in Part 1 exactly as there. Unlike almac11
  (plist cutouts), here `zoom_box_len = 100` matches the 100 pkpc region cutouts
  exactly — the 1→100 kpc aperture ladder is a real mock aperture set.
- Torus arms use a SINGLE sightline (`THETA=[0]` → `i0p0`), the almac11 cost
  decision (~4× cheaper); the 4-sightline block is left commented in the masters
  if sightline-resolved torus SEDs are ever needed.
- Torus + Hopkins arms only for galaxies with ≥1 BH particle in their Stage-0
  region cutout (a galaxy with no BH would just reproduce `dust_on` at 2× cost).
  Cutouts already carry `PartType5` from the 2026-07-28 agn_on re-extraction.

**Downstream**: to pull the new arms through the flux-extraction/CIGALE ladder of
the main notebook, extend its `RUNS` dict with
`'nenkova_i90': dict(run_tag='nenkova_i90', paramf='parameters_master-nenkova-i90.py')`
(etc.) — remembering the torus arms carry only the `i0p0` sightline.


# Part 0 — Configuration

Everything below runs ON the cluster (pd39 kernel). The selection is read from
`powderday_quenched_selection.fits`, written by Part 3 of the main notebook — this
notebook never re-derives the sample.


In [ ]:
# ── Part 0 · configuration ───────────────────────────────────────────────────
import os, glob, re, inspect
from collections import defaultdict

import numpy as np
import h5py
from astropy.table import Table

from simbanator.io.simba import Simulation
from simbanator.sed.makesed import MakeSED

sim = Simulation('cis25')

HOME            = '/mnt/home/glorenzon/analize_simba_cgm'
SED_OUT         = os.path.join(HOME, 'output', sim.name, 'sed_quenched_regions')  # all 6 trees
TABLEDIR        = os.path.join(HOME, 'output', sim.name, 'tables')                # maps
hydro_dir_base  = os.path.join(HOME, 'output', sim.name, 'filtered_particles')    # region cutouts
PARTICLE_PREFIX = sim.file_format.split('_{')[0]        # 'm25n512'
PARTITION       = 'INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL'
R_CUTOUT_KPC    = 100.0                                 # Stage-0 region radius (main nb Part 4)
SELECTION_FITS  = os.path.join(TABLEDIR, 'powderday_quenched_selection.fits')

# ── the six arms: key -> tree tag + paramf + expected knobs (Parts 1/3b assert these)
#    stage=False: existing tree, census only. who='bh': only galaxies with >=1 BH.
ARMS = {
    'dust_on':     dict(tag='dusty_simdust', paramf='parameters_master.py', stage=False,
                        expect=dict(BH_SED='False', dust_grid_type='manual', n_theta=4,
                                    incl=None), who='all'),
    'dust_off':    dict(tag='nodust_1e-12', paramf='parameters_master-nodust.py', stage=False,
                        expect=dict(BH_SED='False', dust_grid_type='dtm', n_theta=4,
                                    incl=None), who='all'),
    'agn_hopkins': dict(tag='dusty_simdust_agn', paramf='parameters_master-agn.py', stage=False,
                        expect=dict(BH_SED='True', BH_model='Hopkins', dust_grid_type='manual',
                                    n_theta=4, incl=None), who='bh'),
    'nenkova_i90': dict(tag='nenkova_i90', paramf='parameters_master-nenkova-i90.py', stage=True,
                        expect=dict(BH_SED='True', BH_model='Nenkova', dust_grid_type='manual',
                                    n_theta=1, incl=90), who='bh'),
    'nenkova_i60': dict(tag='nenkova_i60', paramf='parameters_master-nenkova-i60.py', stage=True,
                        expect=dict(BH_SED='True', BH_model='Nenkova', dust_grid_type='manual',
                                    n_theta=1, incl=60), who='bh'),
    'nenkova_i30': dict(tag='nenkova_i30', paramf='parameters_master-nenkova-i30.py', stage=True,
                        expect=dict(BH_SED='True', BH_model='Nenkova', dust_grid_type='manual',
                                    n_theta=1, incl=30), who='bh'),
}

SEL  = Table.read(SELECTION_FITS)
KEYS = [(int(s), int(g)) for s, g in zip(SEL['snap'], SEL['gal_id'])]
PREFLIGHT_OK = False   # Part 1 sets this; the staging cell refuses to run without it

print(f'sample: {len(SEL)} quenched galaxies over snaps '
      f'{sorted(set(int(s) for s in SEL["snap"]))}  ({SELECTION_FITS})')
for s in sorted(set(int(x) for x in SEL['snap'])):
    m = np.asarray(SEL['snap']) == s
    cls = dict(zip(*np.unique(np.asarray(SEL['agn_class'])[m], return_counts=True)))
    print(f'  snap {s:03d} (z={sim.get_z_from_snap(s):.3f}): {int(m.sum()):3d} gals  {cls}')


# Part 1 — Preflight

Same five checks as the almac11 campaign, read-only: (1a) the powderday copy the
jobs will import, (1b) the gadget2pd BH field-name fix, (2) per-arm master knobs,
(3) the CLUMPY grid the Nenkova arms need, (4) arm homogeneity — only
BH/torus/sightline/dust-off knobs may differ from `dust_on` — and (5) catalogs +
region cutouts (with `PartType5`) for every anchor snap. Nothing stages unless
this passes.


In [ ]:
# ── Part 1 · PREFLIGHT (read-only; no powderday import) ──────────────────────
_fail, _warn = [], []
_SEDDIR = os.path.dirname(inspect.getfile(MakeSED))
print('paramf files resolve against the IMPORTED simbanator:', _SEDDIR)
_repo_sed = os.path.join(HOME, 'simbanator', 'sed')
if os.path.realpath(_SEDDIR) != os.path.realpath(_repo_sed):
    _warn.append(f'imported simbanator ({_SEDDIR}) is not the repo copy ({_repo_sed}); '
                 'sync/reinstall first or the new paramf files may be missing/stale there.')

# 1a. the powderday copy the JOBS will import ---------------------------------
_setup = os.path.join(_SEDDIR, 'cosmology_setup_all_cluster.cis.sh')
_m = re.search(r'PD_FRONT_END=\\?"([^"\\]+)', open(_setup).read()) if os.path.exists(_setup) else None
if _m is None:
    _fail.append(f'could not read PD_FRONT_END out of {_setup}')
    PD_FRONT_END = PD_DIR = None
else:
    PD_FRONT_END = _m.group(1)
    PD_DIR = os.path.dirname(PD_FRONT_END)
    print(f'jobs run       : python {PD_FRONT_END} . parameters_master snapNN_ID')
    if not os.path.exists(PD_FRONT_END):
        _fail.append(f'{PD_FRONT_END} does not exist')

# 1b. BH field names in gadget2pd.py (malformed names crash EVERY BH_SED job) --
_BAD_MARKER = '("bh' + "','"
_WANT = ['coordinates', 'luminosity', 'nu', 'sed']
if PD_DIR:
    _g2p = os.path.join(PD_DIR, 'powderday', 'front_ends', 'gadget2pd.py')
    if not os.path.exists(_g2p):
        _fail.append(f'{_g2p} not found')
    else:
        _src = open(_g2p).read()
        _bad = _src.count(_BAD_MARKER)
        _good = sorted(set(re.findall(r'add_field\(\("bh","(\w+)"\)', _src)))
        print(f'BH field names : malformed {_bad} | correct {_good}')
        if _bad or _good != _WANT:
            _fail.append('gadget2pd.py BH field names are malformed — every BH_SED=True job '
                         'would raise ValueError. Re-apply the fix (*.pre-bhfix-20260730.bak).')
    _sc = os.path.join(PD_DIR, 'powderday', 'source_creation.py')
    if os.path.exists(_sc) and 'reg["bh","sed"]' not in open(_sc).read().replace("'", '"'):
        _fail.append('source_creation.py no longer indexes reg["bh","sed"] — field-name '
                     'contract changed; re-derive expectations before trusting 1b.')

# 2. per-arm parameter masters ------------------------------------------------
def _parse_master(path):
    src = open(path).read()
    v = {}
    for key in ('BH_SED', 'BH_var', 'dust_grid_type', 'BH_model',
                'dusttometals_ratio', 'n_photons_raytracing_dust'):
        m = re.search(rf'(?m)^{key}\s*=\s*(.+?)\s*(#.*)?$', src)
        v[key] = m.group(1).strip().strip('"\'') if m else None
    m = re.search(r'(?m)^nenkova_params\s*=\s*\[([^\]]+)\]', src)
    v['incl'] = int(float(m.group(1).split(',')[2])) if m else None
    m = re.search(r'(?m)^THETA\s*=\s*\[([^\]]+)\]', src)
    v['n_theta'] = len(m.group(1).split(',')) if m else None
    return v, src

print('\nper-arm parameter masters:')
_master_src = {}
for arm, spec in ARMS.items():
    p = os.path.join(_SEDDIR, spec['paramf'])
    if not os.path.exists(p):
        _fail.append(f"{arm}: {spec['paramf']} not found in {_SEDDIR}")
        continue
    v, _master_src[arm] = _parse_master(p)
    exp = spec['expect']
    bad = [k for k, want in exp.items() if want is not None and v.get(k) != str(want)
           and v.get(k) != want]
    print(f"  {arm:12s} BH_SED={v['BH_SED']:5s} model={str(v['BH_model']):8s} "
          f"i={str(v['incl']):4s} n_theta={v['n_theta']} grid={v['dust_grid_type']} "
          f"-> {'PASS' if not bad else 'FAIL ' + str(bad)}")
    if bad:
        _fail.append(f"{arm}: {spec['paramf']} knobs {bad} do not match {exp}")
    if v['BH_SED'] == 'True' and v['BH_var'] != 'False':
        _fail.append(f'{arm}: BH_var must be False to keep L_AGN = 0.1*Mdot_SIMBA*c^2')

# 3. CLUMPY grid needed by the Nenkova arms -----------------------------------
_clumpy = os.path.join(os.environ.get('POWDERDAY_ROOT', os.path.expanduser('~')),
                       'powderday', 'agn_models', 'clumpy_models_201410_tvavg.hdf5')
if os.path.exists(_clumpy):
    print(f'\nCLUMPY grid    : {_clumpy}  ({os.path.getsize(_clumpy) / 1e9:.2f} GB)')
else:
    _fail.append(f'CLUMPY grid missing: {_clumpy} — the three Nenkova arms cannot run.')

# 4. arm homogeneity: only BH/torus/sightline/dust-off knobs may differ -------
_ALLOWED = {'BH_SED', 'BH_eta', 'BH_model', 'BH_modelfile', 'BH_var', 'nenkova_params',
            'THETA', 'PHI', 'dust_grid_type', 'dusttometals_ratio', 'n_photons_raytracing_dust'}
_ref = [l for l in _master_src.get('dust_on', '').splitlines()
        if l.strip() and not l.strip().startswith('#')]
for arm, src in _master_src.items():
    if arm == 'dust_on':
        continue
    _b = [l for l in src.splitlines() if l.strip() and not l.strip().startswith('#')]
    _keys = set()
    for l in sorted(set(_ref) ^ set(_b)):
        mk = re.match(r'\s*(\w+)\s*=', l)
        _keys.add(mk.group(1) if mk else l.strip())
    _extra = sorted(_keys - _ALLOWED)
    print(f'  {arm:12s} differs from dust_on in: {sorted(_keys)}')
    if _extra:
        _fail.append(f'{arm}: differs from dust_on outside the allowed knobs ({_extra}) — '
                     'arm differences would no longer isolate torus/dust effects.')

# 5. catalogs + region cutouts (with PartType5) for every anchor snap ---------
print()
for s in sorted(set(k[0] for k in KEYS)):
    _cat = os.path.exists(sim.get_caesar_file(s))
    _cdir = os.path.join(hydro_dir_base, f'snap_{s:03d}')
    _cuts = sorted(glob.glob(os.path.join(_cdir, f'{PARTICLE_PREFIX}_snap{s:03d}_gal*.h5')))
    _p5 = None
    if _cuts:
        with h5py.File(_cuts[0], 'r') as f:
            _p5 = 'PartType5' in f
    print(f'  snap {s:03d}: catalog {_cat} | cutouts {len(_cuts)} | PartType5 in first: {_p5}')
    if not _cat:
        _fail.append(f'snap {s}: no caesar catalog')
    if not _cuts:
        _fail.append(f'snap {s}: no region cutouts — run Part 4 of the main notebook first')
    elif not _p5:
        _fail.append(f'snap {s}: cutouts lack PartType5 — re-run main-notebook Part 4 '
                     '(EXTRACT_OVERWRITE=True) before any BH_SED arm')

print('\n' + '=' * 78)
for w in _warn:
    print('WARN:', w)
if _fail:
    for f_ in _fail:
        print('FAIL:', f_)
    raise RuntimeError('Preflight failed — do NOT run the staging cell.')
PREFLIGHT_OK = True
print('PREFLIGHT PASSED — all six arms are wired up.')


# Part 2 — BH census + reconcile vs the existing trees

Counts the BH particles in every Stage-0 region cutout (the `who='bh'` gate), then
takes a `sed_is_complete` census: the three existing trees are never restaged —
incomplete SEDs there are a *submission* problem, flagged for their own
`submit_all_snaps.sh`. Only Nenkova SEDs that do not already exist land in the
`RUN` lists that Part 3 stages. Writes `tables/m25_arms_reuse_map.fits`.


In [ ]:
# ── Part 2 · BH census + reconcile ───────────────────────────────────────────
def rtout_path_tag(tag, snap, gid, root=None):
    return os.path.join(root or SED_OUT, tag, 'powderday_sed_out', f'snap_{int(snap):03d}',
                        f'gal_{int(gid)}', f'snap{int(snap):03d}.galaxy{int(gid):06d}.rtout.sed')

def sed_is_complete(path):
    # True only if the RT actually FINISHED (Peeled SED groups are written last).
    if not (path and os.path.exists(path)):
        return False
    try:
        with h5py.File(path, 'r') as f:
            g = f.get('Peeled')
            return bool(g) and any('seds' in g[k] for k in g.keys())
    except OSError:
        return False

def cutout_file(snap, gid):
    return os.path.join(hydro_dir_base, f'snap_{int(snap):03d}',
                        f'{PARTICLE_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5')

def n_bh(snap, gid):
    pf = cutout_file(snap, gid)
    if not os.path.exists(pf):
        return -1
    with h5py.File(pf, 'r') as f:
        return int(f['PartType5/BH_Mass'].shape[0]) if 'PartType5' in f else 0

NBH = {k: n_bh(*k) for k in KEYS}
_missing = [k for k, n in NBH.items() if n < 0]
if _missing:
    raise RuntimeError(f'{len(_missing)} cutouts missing (e.g. {_missing[:3]}) — '
                       're-run Part 4 of the main notebook first.')
BH_HOSTS = [k for k in KEYS if NBH[k] > 0]
print(f'BH census: {len(BH_HOSTS)}/{len(KEYS)} galaxies host >=1 BH in the 100 pkpc cutout '
      f'(max {max(NBH.values())} BHs; the no-BH rest only get dust_on/dust_off)')

CENSUS_ROWS = []                       # (arm, snap, gid, tree_tag, path, complete)
RUN = {arm: [] for arm in ARMS}
for arm, spec in ARMS.items():
    for key in KEYS:
        if spec['who'] == 'bh' and NBH[key] == 0:
            continue
        snap, gid = key
        p = rtout_path_tag(spec['tag'], snap, gid)
        ok = sed_is_complete(p)
        if spec['stage'] and not ok:
            RUN[arm].append(key)       # to stage in Part 3
        else:
            CENSUS_ROWS.append((arm, snap, gid, spec['tag'], p, ok))

R = Table(rows=CENSUS_ROWS or None,
          names=('ARM', 'SNAPSHOT', 'GROUPID_SNAPSHOT', 'TREE_TAG', 'RTOUT_PATH', 'COMPLETE'))
R.write(os.path.join(TABLEDIR, 'm25_arms_reuse_map.fits'), overwrite=True)

print(f'\n{"arm":12s} {"to run":>7s} {"complete":>9s} {"pending":>8s}   note')
for arm, spec in ARMS.items():
    done = sum(1 for r_ in CENSUS_ROWS if r_[0] == arm and r_[5])
    pend = sum(1 for r_ in CENSUS_ROWS if r_[0] == arm and not r_[5])
    note = '' if spec['stage'] else 'existing tree — pending = resubmit its submit_all_snaps.sh'
    print(f'{arm:12s} {len(RUN[arm]):7d} {done:9d} {pend:8d}   {note}')

# rough cost, in units of one 4-sightline AGN-off (dust_on) galaxy:
#   nenkova ~2.0 (8x AGN cost / 4 for the single sightline) — almac11 calibration
_units = sum(len(RUN[a]) for a in ('nenkova_i90', 'nenkova_i60', 'nenkova_i30')) * 2.0
print(f'\nestimated NEW cost ~ {_units:.0f} dust_on-equivalent galaxy runs '
      f'(CEERS benchmark: ~9 cpu-h each at 1e6 photons)')


# Part 3 — Stage the three Nenkova arms

`MakeSED` in **region** mode (radius = 100 pkpc), identical to the existing trees;
per-arm `selection_file` so the target-selection h5 files never collide with
`selection_m25_quenched`. Gated on the Part 1 preflight.


In [ ]:
# ── Part 3 · stage the Nenkova arms ──────────────────────────────────────────
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight did not pass.')

for arm, spec in ARMS.items():
    if not spec['stage']:
        continue
    keys = RUN[arm]
    if not keys:
        print(f'{arm:12s}: nothing to run (all complete) — skipped')
        continue
    snaps = np.array([k[0] for k in keys], int)
    ids = np.array([k[1] for k in keys], int)
    ms = MakeSED(sim, nnodes=1, model_run_name=spec['tag'], hydro_dir_base=hydro_dir_base,
                 selection_file=f'rt_m25_{arm}', output_dir=SED_OUT, run_tag=spec['tag'])
    ms.selection_gals(snaps=snaps, galaxyID=ids)
    ms.create_master('cluster', 'region', radius=R_CUTOUT_KPC, partition=PARTITION,
                     prefix=PARTICLE_PREFIX, paramf=spec['paramf'], snaps_to_run=None)
    print(f'{arm:12s}: staged {len(keys)} galaxies over snaps '
          f'{sorted(set(snaps.tolist()))} -> {os.path.join(SED_OUT, spec["tag"])}')


# Part 3b — Verify the staged masters, then submit

Re-parses every staged `snap_*/parameters_master.py` and compares `ids.txt`
against the `RUN` lists before printing the `sbatch` commands. Calibrate ONE
galaxy first (submit one array index) before unleashing a whole arm.


In [ ]:
# ── Part 3b · verify staged parameter files + print submit commands ──────────
_all_ok = True
for arm, spec in ARMS.items():
    if not spec['stage'] or not RUN[arm]:
        continue
    exp = spec['expect']
    print(f'=== {arm} (expect BH_SED={exp["BH_SED"]}, i={exp["incl"]}, '
          f'n_theta={exp["n_theta"]}, grid={exp["dust_grid_type"]}) ===')
    for s in sorted(set(k[0] for k in RUN[arm])):
        jdir = os.path.join(SED_OUT, spec['tag'], 'powderday_sed_out', f'snap_{s:03d}')
        if not os.path.isdir(jdir):
            jdir = os.path.join(SED_OUT, spec['tag'], 'powderday_sed_out', f'snap_{s}')
        pm = os.path.join(jdir, 'parameters_master.py')
        if not os.path.exists(pm):
            print(f'  snap {s}: !! no parameters_master.py'); _all_ok = False; continue
        v, _ = _parse_master(pm)
        nids = (sum(1 for _ in open(os.path.join(jdir, 'ids.txt')))
                if os.path.exists(os.path.join(jdir, 'ids.txt')) else 0)
        nexp = sum(1 for k in RUN[arm] if k[0] == s)
        bad = [k for k, want in exp.items() if want is not None
               and v.get(k) != str(want) and v.get(k) != want]
        print(f'  snap {s:03d}: BH_SED={v["BH_SED"]} model={v["BH_model"]} i={v["incl"]} '
              f'n_theta={v["n_theta"]} grid={v["dust_grid_type"]} | ids {nids} (expect {nexp}) '
              f'-> {"OK" if not bad and nids == nexp else "FAIL"}')
        if bad or nids != nexp:
            _all_ok = False

print('\n' + ('staging verified' if _all_ok else '!! STAGING PROBLEM — do not submit'))
print('\n── submit the NEW arms with (calibrate ONE galaxy first!) ──')
for arm in ('nenkova_i90', 'nenkova_i60', 'nenkova_i30'):
    for s in sorted(set(k[0] for k in RUN[arm])):
        jdir = os.path.join(SED_OUT, ARMS[arm]['tag'], 'powderday_sed_out', f'snap_{s:03d}')
        jobs = sorted(glob.glob(os.path.join(jdir, 'master.snap*.job')))
        if jobs:
            print(f'cd {jdir} && sbatch {os.path.basename(jobs[-1])}')
        elif RUN[arm]:
            print(f'!! no master.snap*.job in {jdir} — staging failed?')

# existing arms: anything pending there is resubmitted from its OWN tree
_pending = defaultdict(int)
for r_ in CENSUS_ROWS:
    if not r_[5]:
        _pending[r_[0]] += 1
for arm, n in sorted(_pending.items()):
    tree = os.path.join(SED_OUT, ARMS[arm]['tag'], 'powderday_sed_out')
    print(f'\n{arm}: {n} SEDs incomplete in the EXISTING tree -> '
          f'cd {tree} && bash submit_all_snaps.sh')


# Part 4 — Source-map census

Fresh `sed_is_complete` sweep over all six arms → `tables/m25_arms_source_map.fits`
(`ARM, SNAPSHOT, GROUPID_SNAPSHOT, TREE_TAG, RTOUT_PATH, COMPLETE`). Re-run as the
RT progresses; downstream analysis should read only `COMPLETE` rows.


In [ ]:
# ── Part 4 · census -> m25_arms_source_map.fits ──────────────────────────────
rows = []
for arm, spec in ARMS.items():
    for key in KEYS:
        if spec['who'] == 'bh' and NBH[key] == 0:
            continue
        snap, gid = key
        p = rtout_path_tag(spec['tag'], snap, gid)
        rows.append((arm, snap, gid, spec['tag'], p, sed_is_complete(p)))

S = Table(rows=rows, names=('ARM', 'SNAPSHOT', 'GROUPID_SNAPSHOT', 'TREE_TAG',
                            'RTOUT_PATH', 'COMPLETE'))
S.write(os.path.join(TABLEDIR, 'm25_arms_source_map.fits'), overwrite=True)

print(f'{"arm":12s} {"planned":>8s} {"complete":>9s}')
for arm in ARMS:
    m_ = np.asarray(S['ARM']) == arm
    print(f'{arm:12s} {int(m_.sum()):8d} {int(np.asarray(S["COMPLETE"])[m_].sum()):9d}')
print(f"\nsource map -> {os.path.join(TABLEDIR, 'm25_arms_source_map.fits')}")
